# AEGIS — Adversarial Prompt Classification
## Exploratory Data Analysis & Model Training

**Project:** Adversarial Enforcement and Guardrail Interception System (AEGIS)  
**Task:** 5-class text classification of LLM prompts in a healthcare context  

| Label | Class | Description |
|-------|-------|-------------|
| 0 | `benign` | Legitimate clinical / biomedical queries |
| 1 | `direct_injection` | Direct prompt-injection attacks |
| 2 | `indirect_injection` | Malicious instructions hidden in context |
| 3 | `jailbreak` | Role-play / persona-based restriction bypass |
| 4 | `phi_extraction` | Attempts to extract Protected Health Information |

---

## 0 — Google Colab Setup

> **Run the cell below first** if you are on Google Colab. It clones the repository, installs all dependencies (PyTorch, Transformers, scikit-learn, etc.), and runs the preprocessing pipeline to generate training data from public HuggingFace datasets.
>
> On Colab, select **Runtime → Change runtime type → T4 GPU** for faster BERT-Tiny training.

In [ ]:
import subprocess, sys, os
from pathlib import Path

COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if COLAB:
    REPO_ROOT = "/content/aegis"

    if not os.path.exists(REPO_ROOT):
        subprocess.run(
            ["git", "clone", "https://github.com/espirado/Aegis.git", REPO_ROOT],
            check=True,
        )
        print("Repository cloned.")
    else:
        print("Repository already cloned.")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", f"{REPO_ROOT}/ml/requirements.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "wordcloud", "datasets"],
        check=True,
    )
    print("Dependencies installed.")

    if not os.path.exists(f"{REPO_ROOT}/data/processed/train.jsonl"):
        subprocess.run(
            [sys.executable, "scripts/data/preprocess.py",
             "--output-dir", "data/processed"],
            cwd=REPO_ROOT,
            check=True,
        )
        print("Preprocessing complete.")
    else:
        print("Processed data already exists.")
else:
    REPO_ROOT = str(Path("../..").resolve())
    print(f"Running locally — REPO_ROOT = {REPO_ROOT}")

print(f"REPO_ROOT = {REPO_ROOT}")

## 1 — Import Libraries & Configuration

In [ ]:
import json
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

SEED = 42
np.random.seed(SEED)

CLASS_NAMES = [
    "benign",
    "direct_injection",
    "indirect_injection",
    "jailbreak",
    "phi_extraction",
]
NUM_CLASSES = len(CLASS_NAMES)

DATA_DIR = Path(REPO_ROOT) / "data" / "processed"
MODEL_DIR = Path(REPO_ROOT) / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Classes: {CLASS_NAMES}")
print(f"Data dir: {DATA_DIR.resolve()}")

## 2 — Load and Inspect JSONL Data

In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    """Read a JSONL file into a DataFrame."""
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)


df_train = load_jsonl(DATA_DIR / "train.jsonl")
df_val = load_jsonl(DATA_DIR / "val.jsonl")
df_test = load_jsonl(DATA_DIR / "test.jsonl")

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f"Train : {len(df_train):,}")
print(f"Val   : {len(df_val):,}")
print(f"Test  : {len(df_test):,}")
print(f"Total : {len(df_all):,}")
print()
df_all.head(10)

In [ ]:
df_all.info()
print()
df_all.describe(include="all")

In [ ]:
df_all["label_name"] = df_all["label"].map(dict(enumerate(CLASS_NAMES)))
df_train["label_name"] = df_train["label"].map(dict(enumerate(CLASS_NAMES)))
df_val["label_name"] = df_val["label"].map(dict(enumerate(CLASS_NAMES)))
df_test["label_name"] = df_test["label"].map(dict(enumerate(CLASS_NAMES)))

print("Missing values:")
print(df_all.isnull().sum())
print(f"\nDuplicate texts: {df_all['text'].duplicated().sum():,}")

## 3 — Label Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
label_counts = df_all["label_name"].value_counts().reindex(CLASS_NAMES)
ax = label_counts.plot.bar(ax=axes[0], color=sns.color_palette("muted", NUM_CLASSES), edgecolor="black")
axes[0].set_title("Label Distribution (Counts)", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
for bar in ax.patches:
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{int(bar.get_height()):,}",
        ha="center", va="bottom", fontsize=10,
    )

# Proportions
label_pct = (label_counts / label_counts.sum()) * 100
axes[1].pie(
    label_pct,
    labels=[f"{n}\n({p:.1f}%)" for n, p in zip(CLASS_NAMES, label_pct)],
    colors=sns.color_palette("muted", NUM_CLASSES),
    startangle=140,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5},
)
axes[1].set_title("Label Distribution (Proportions)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Per-split distribution
split_dist = pd.DataFrame({
    "Train": df_train["label_name"].value_counts().reindex(CLASS_NAMES),
    "Val": df_val["label_name"].value_counts().reindex(CLASS_NAMES),
    "Test": df_test["label_name"].value_counts().reindex(CLASS_NAMES),
})
split_dist["Total"] = split_dist.sum(axis=1)
split_dist

## 4 — Text Length Analysis with Outlier Detection

In [ ]:
df_all["text_len"] = df_all["text"].str.len()
df_all["word_count"] = df_all["text"].str.split().str.len()

print("Character-length statistics:")
print(df_all["text_len"].describe().to_string())
print(f"\nWord-count statistics:")
print(df_all["word_count"].describe().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram per class
for i, cls in enumerate(CLASS_NAMES):
    subset = df_all[df_all["label_name"] == cls]
    axes[0].hist(subset["word_count"], bins=50, alpha=0.6, label=cls)
axes[0].set_title("Word Count Distribution by Class", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, df_all["word_count"].quantile(0.99))

# Box plot
sns.boxplot(
    data=df_all,
    x="label_name",
    y="word_count",
    order=CLASS_NAMES,
    ax=axes[1],
    showfliers=False,
)
axes[1].set_title("Word Count Box Plot (outliers hidden)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Word Count")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# IQR-based outlier detection
Q1 = df_all["word_count"].quantile(0.25)
Q3 = df_all["word_count"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_all[(df_all["word_count"] < lower_bound) | (df_all["word_count"] > upper_bound)]
print(f"IQR range: [{lower_bound:.0f}, {upper_bound:.0f}]")
print(f"Outliers: {len(outliers):,} ({len(outliers)/len(df_all)*100:.1f}%)")
print(f"\nOutlier breakdown by class:")
print(outliers["label_name"].value_counts())

## 5 — Text Cleaning Pipeline

In [ ]:
def clean_text(text: str) -> str:
    """Normalize text for model input while preserving semantic structure."""
    text = text.strip()
    text = re.sub(r"http\S+|www\.\S+", "[URL]", text)
    text = re.sub(r"\S+@\S+", "[EMAIL]", text)
    text = re.sub(r"\b\d{3}[-.]?\d{2}[-.]?\d{4}\b", "[SSN]", text)
    text = re.sub(r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b", "[PHONE]", text)
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


# Preview before/after
sample_idx = df_all[df_all["label"] == 2].index[0]
sample_raw = df_all.loc[sample_idx, "text"]
sample_clean = clean_text(sample_raw)

print("BEFORE:")
print(sample_raw[:300])
print("\nAFTER:")
print(sample_clean[:300])

In [ ]:
df_all["text_clean"] = df_all["text"].apply(clean_text)
df_train["text_clean"] = df_train["text"].apply(clean_text)
df_val["text_clean"] = df_val["text"].apply(clean_text)
df_test["text_clean"] = df_test["text"].apply(clean_text)

df_all["clean_len"] = df_all["text_clean"].str.len()
len_reduction = (1 - df_all["clean_len"].sum() / df_all["text_len"].sum()) * 100
print(f"Total character reduction after cleaning: {len_reduction:.2f}%")
print(f"Empty after cleaning: {(df_all['text_clean'].str.len() == 0).sum()}")

## 6 — Tokenization Analysis & Max Length Selection

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "prajjwal1/bert-tiny"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {MODEL_NAME}  |  Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
token_lengths = df_all["text_clean"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)

percentiles = [50, 75, 90, 95, 99]
print("Token length percentiles:")
for p in percentiles:
    val = np.percentile(token_lengths, p)
    print(f"  P{p:>2}: {val:.0f} tokens")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(token_lengths, bins=80, edgecolor="black", alpha=0.7)
for p in [90, 95, 99]:
    val = np.percentile(token_lengths, p)
    ax.axvline(val, color="red", linestyle="--", alpha=0.7)
    ax.text(val + 2, ax.get_ylim()[1] * 0.9, f"P{p}={val:.0f}", fontsize=9, color="red")

MAX_LEN = 128
ax.axvline(MAX_LEN, color="green", linewidth=2, label=f"MAX_LEN = {MAX_LEN}")
ax.set_title("Token Length Distribution (BERT-Tiny Tokenizer)", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Tokens")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.show()

coverage = (token_lengths <= MAX_LEN).mean() * 100
print(f"\nMAX_LEN = {MAX_LEN} covers {coverage:.1f}% of samples without truncation.")

## 7 — Class Weight Computation for Imbalanced Data

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=df_train["label"].values,
)

class_weight_dict = dict(zip(CLASS_NAMES, class_weights))
print("Computed class weights (sklearn balanced):")
for name, w in class_weight_dict.items():
    print(f"  {name:<25s}: {w:.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, class_weights, color=sns.color_palette("muted", NUM_CLASSES), edgecolor="black")
ax.axhline(1.0, color="gray", linestyle="--", label="Uniform weight (1.0)")
ax.set_title("Class Weights for Imbalanced Training", fontsize=14, fontweight="bold")
ax.set_ylabel("Weight")
ax.legend()
for bar, w in zip(bars, class_weights):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{w:.2f}",
            ha="center", va="bottom", fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 8 — Word Clouds & Top Unigrams per Class

In [ ]:
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print("wordcloud not installed — pip install wordcloud; skipping word clouds.")

if HAS_WORDCLOUD:
    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(24, 5))
    for i, cls in enumerate(CLASS_NAMES):
        corpus = " ".join(df_all[df_all["label_name"] == cls]["text_clean"])
        wc = WordCloud(
            width=400, height=300,
            background_color="white",
            max_words=80,
            colormap="viridis",
            random_state=SEED,
        ).generate(corpus)
        axes[i].imshow(wc, interpolation="bilinear")
        axes[i].set_title(cls, fontsize=12, fontweight="bold")
        axes[i].axis("off")
    plt.suptitle("Word Clouds per Class", fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(24, 5))

for i, cls in enumerate(CLASS_NAMES):
    texts = df_all[df_all["label_name"] == cls]["text_clean"]
    cv = CountVectorizer(stop_words="english", max_features=15)
    counts = cv.fit_transform(texts)
    freq = dict(zip(cv.get_feature_names_out(), counts.sum(axis=0).A1))
    freq = dict(sorted(freq.items(), key=lambda x: x[1], reverse=True))

    axes[i].barh(list(freq.keys())[::-1], list(freq.values())[::-1], color=sns.color_palette("muted")[i])
    axes[i].set_title(cls, fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Count")

plt.suptitle("Top 15 Unigrams per Class (stop words removed)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 9 — TF-IDF Vectorization & Dimensionality Reduction (t-SNE)

In [ ]:
N_SAMPLE = min(3000, len(df_all))
df_sample = df_all.sample(n=N_SAMPLE, random_state=SEED)

tfidf = TfidfVectorizer(max_features=5000, stop_words="english", sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df_sample["text_clean"])
print(f"TF-IDF matrix shape: {X_tfidf.shape}")

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000, learning_rate="auto")
X_2d = tsne.fit_transform(X_tfidf.toarray())

fig, ax = plt.subplots(figsize=(10, 8))
palette = sns.color_palette("muted", NUM_CLASSES)
for i, cls in enumerate(CLASS_NAMES):
    mask = df_sample["label_name"].values == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], s=12, alpha=0.6, label=cls, color=palette[i])

ax.set_title("t-SNE of TF-IDF Vectors (5 000 features)", fontsize=14, fontweight="bold")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(title="Class", fontsize=9, markerscale=2)
plt.tight_layout()
plt.show()

## 10 — Train / Validation / Test Split Verification

In [ ]:
total = len(df_train) + len(df_val) + len(df_test)

split_summary = pd.DataFrame({
    "Split": ["Train", "Val", "Test"],
    "Samples": [len(df_train), len(df_val), len(df_test)],
    "Ratio": [len(df_train)/total, len(df_val)/total, len(df_test)/total],
})
split_summary["Ratio"] = split_summary["Ratio"].map("{:.1%}".format)
print(split_summary.to_string(index=False))

# Verify no leakage between splits
train_texts = set(df_train["text"])
val_texts = set(df_val["text"])
test_texts = set(df_test["text"])

leak_tv = train_texts & val_texts
leak_tt = train_texts & test_texts
leak_vt = val_texts & test_texts

print(f"\nData leakage check:")
print(f"  Train ∩ Val  : {len(leak_tv)}")
print(f"  Train ∩ Test : {len(leak_tt)}")
print(f"  Val   ∩ Test : {len(leak_vt)}")
if len(leak_tv) + len(leak_tt) + len(leak_vt) == 0:
    print("  ✓ No data leakage detected.")

## 11 — Baseline Model: Logistic Regression on TF-IDF

In [ ]:
tfidf_baseline = TfidfVectorizer(max_features=10000, stop_words="english", sublinear_tf=True)
X_train_tfidf = tfidf_baseline.fit_transform(df_train["text_clean"])
X_val_tfidf = tfidf_baseline.transform(df_val["text_clean"])
X_test_tfidf = tfidf_baseline.transform(df_test["text_clean"])

y_train = df_train["label"].values
y_val = df_val["label"].values
y_test = df_test["label"].values

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    multi_class="multinomial",
    random_state=SEED,
    C=1.0,
)
lr_model.fit(X_train_tfidf, y_train)

y_pred_val = lr_model.predict(X_val_tfidf)
y_pred_test = lr_model.predict(X_test_tfidf)

print("=" * 60)
print("BASELINE — Logistic Regression + TF-IDF")
print("=" * 60)
print("\nValidation Set:")
print(classification_report(y_val, y_pred_val, target_names=CLASS_NAMES, zero_division=0))
print("Test Set:")
print(classification_report(y_test, y_pred_test, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_true, y_pred, title in [
    (axes[0], y_val, y_pred_val, "Baseline — Validation"),
    (axes[1], y_test, y_pred_test, "Baseline — Test"),
]:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Logistic Regression Confusion Matrices", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 12 — BERT-Tiny Fine-Tuning with Weighted Loss

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
class PromptDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            texts.tolist(),
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "label": self.labels[idx],
        }


print("Tokenizing datasets...")
train_ds = PromptDataset(df_train["text_clean"], df_train["label"].values, tokenizer, MAX_LEN)
val_ds = PromptDataset(df_val["text_clean"], df_val["label"].values, tokenizer, MAX_LEN)
test_ds = PromptDataset(df_test["text_clean"], df_test["label"].values, tokenizer, MAX_LEN)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES
).to(device)

weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=ids, attention_mask=mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=ids, attention_mask=mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = probs.argmax(dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_fpr": []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    scheduler.step()

    val_preds, val_labels, _ = evaluate_model(model, val_loader, device)
    val_acc = (val_preds == val_labels).mean()

    benign_mask = val_labels == 0
    val_fpr = (val_preds[benign_mask] != 0).mean() if benign_mask.sum() > 0 else 0.0

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_fpr"].append(val_fpr)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict().copy()
        marker = " ★"

    print(
        f"Epoch {epoch:>2}/{EPOCHS}  "
        f"loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
        f"val_acc={val_acc:.4f}  val_fpr={val_fpr:.4f}{marker}"
    )

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history["train_loss"], "o-", label="Train Loss")
axes[0].set_title("Training Loss", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(epochs_range, history["train_acc"], "o-", label="Train Acc")
axes[1].plot(epochs_range, history["val_acc"], "s-", label="Val Acc")
axes[1].set_title("Accuracy", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].plot(epochs_range, history["val_fpr"], "D-", color="red", label="Val Benign FPR")
axes[2].axhline(0.01, color="gray", linestyle="--", label="Target (1%)")
axes[2].set_title("Benign False Positive Rate", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("FPR")
axes[2].legend()

plt.suptitle("BERT-Tiny Training Curves", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 13 — Model Evaluation: Confusion Matrix & Classification Report

In [ ]:
model.load_state_dict(best_state)

test_preds, test_labels, test_probs = evaluate_model(model, test_loader, device)
test_acc = (test_preds == test_labels).mean()

benign_mask = test_labels == 0
test_fpr = (test_preds[benign_mask] != 0).mean() if benign_mask.sum() > 0 else 0.0

print("=" * 60)
print("BERT-TINY — Test Set Evaluation")
print("=" * 60)
print(f"  Accuracy:   {test_acc:.4f}")
print(f"  Benign FPR: {test_fpr:.4f}  (target: < 0.01)")
print()
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
cm = confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES)))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=axes[0], cmap="Blues", values_format="d", colorbar=False)
axes[0].set_title("Test Set — Counts", fontsize=13, fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp_norm = ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES)
disp_norm.plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False)
axes[1].set_title("Test Set — Normalized", fontsize=13, fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("BERT-Tiny Confusion Matrices", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Expected Calibration Error
def compute_ece(probs, labels, n_bins=10):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += mask.sum() / len(labels) * abs(bin_acc - bin_conf)
    return ece


ece = compute_ece(test_probs, test_labels)
print(f"Expected Calibration Error (ECE): {ece:.4f}  (target: < 0.05)")

# Reliability diagram
fig, ax = plt.subplots(figsize=(6, 6))
confidences = test_probs.max(axis=1)
predictions = test_probs.argmax(axis=1)
accuracies = (predictions == test_labels).astype(float)

n_bins = 10
bin_boundaries = np.linspace(0, 1, n_bins + 1)
bin_accs, bin_confs = [], []
for i in range(n_bins):
    mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
    if mask.sum() == 0:
        bin_accs.append(0)
        bin_confs.append((bin_boundaries[i] + bin_boundaries[i+1]) / 2)
    else:
        bin_accs.append(accuracies[mask].mean())
        bin_confs.append(confidences[mask].mean())

ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.7, edgecolor="black", label="Model")
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Mean Predicted Confidence")
ax.set_ylabel("Fraction of Positives")
ax.set_title(f"Reliability Diagram (ECE = {ece:.4f})", fontsize=13, fontweight="bold")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison: Baseline vs BERT-Tiny
baseline_report = classification_report(y_test, y_pred_test, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
bert_report = classification_report(test_labels, test_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)

comparison = pd.DataFrame({
    "Class": CLASS_NAMES + ["macro avg"],
    "LR F1": [baseline_report[c]["f1-score"] for c in CLASS_NAMES] + [baseline_report["macro avg"]["f1-score"]],
    "BERT F1": [bert_report[c]["f1-score"] for c in CLASS_NAMES] + [bert_report["macro avg"]["f1-score"]],
})
comparison["Δ F1"] = comparison["BERT F1"] - comparison["LR F1"]
comparison[["LR F1", "BERT F1", "Δ F1"]] = comparison[["LR F1", "BERT F1", "Δ F1"]].round(4)
print("Baseline (LR+TF-IDF) vs BERT-Tiny:")
print(comparison.to_string(index=False))

## 14 — Export Trained Model & Tokenizer

In [ ]:
model.eval()
model.cpu()

dummy = tokenizer(
    "Example clinical query about patient treatment",
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)

onnx_path = MODEL_DIR / "classifier_v1.onnx"

torch.onnx.export(
    model,
    (dummy["input_ids"], dummy["attention_mask"]),
    str(onnx_path),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size"},
        "attention_mask": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    opset_version=14,
)
print(f"ONNX model exported → {onnx_path.resolve()}")

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(onnx_path))
ort_inputs = {
    "input_ids": dummy["input_ids"].numpy(),
    "attention_mask": dummy["attention_mask"].numpy(),
}
ort_outputs = session.run(None, ort_inputs)
print(f"ONNX validation passed. Output shape: {ort_outputs[0].shape}")
print(f"Predicted class for dummy input: {CLASS_NAMES[ort_outputs[0].argmax()]}")

In [ ]:
import time

metrics = {
    "model_name": MODEL_NAME,
    "epochs": EPOCHS,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "best_val_acc": float(best_val_acc),
    "test_acc": float(test_acc),
    "test_fpr_benign": float(test_fpr),
    "test_ece": float(ece),
    "test_report": bert_report,
    "baseline_test_report": baseline_report,
    "class_weights": {n: float(w) for n, w in class_weight_dict.items()},
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}

metrics_path = MODEL_DIR / "training_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved → {metrics_path.resolve()}")

print("\n" + "=" * 60)
print("AEGIS CLASSIFIER TRAINING COMPLETE")
print("=" * 60)
print(f"  Model:              {MODEL_NAME}")
print(f"  Best val accuracy:  {best_val_acc:.4f}")
print(f"  Test accuracy:      {test_acc:.4f}")
print(f"  Benign FPR:         {test_fpr:.4f}  (target: < 0.01)")
print(f"  ECE:                {ece:.4f}  (target: < 0.05)")
print(f"  ONNX model:         {onnx_path}")
print("=" * 60)